In [18]:
from dotenv import load_dotenv
import requests, json, time, os

load_dotenv()

api_key = os.getenv("football_data_api_key")

In [19]:
import requests, json, time, os

url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

for season in [2023, 2024, 2025]:
    path = f"data/raw/matches_{season}.json"
    if os.path.exists(path):          # past seasons never change, skip refetch
        continue
    r = requests.get(url, headers=headers, params={"season": season})
    print(season, r.status_code)
    r.raise_for_status()
    with open(path, "w") as f:
        json.dump(r.json(), f)
    time.sleep(7)                     # stay under 10 calls/min

In [20]:
import duckdb

con = duckdb.connect()

con.sql("""
    CREATE TABLE matches_clean AS
    SELECT
        m.season.id AS season_id,
        m.matchday AS match_day,
        m.id AS match_id,
        m.utcDate AS date,
        m.homeTeam.name AS home_team,
        m.awayTeam.name AS away_team,
        m.score.winner AS winner
    FROM (
        SELECT unnest(matches) AS m
        FROM read_json_auto('data/raw/matches_*.json')
    )

""")

con.sql("SELECT * FROM matches_clean").show()

┌───────────┬───────────┬──────────┬─────────────────────┬───────────────────────────┬────────────────────────────┬───────────┐
│ season_id │ match_day │ match_id │        date         │         home_team         │         away_team          │  winner   │
│   int64   │   int64   │  int64   │      timestamp      │          varchar          │          varchar           │  varchar  │
├───────────┼───────────┼──────────┼─────────────────────┼───────────────────────────┼────────────────────────────┼───────────┤
│      1564 │         1 │   435943 │ 2023-08-11 19:00:00 │ Burnley FC                │ Manchester City FC         │ AWAY_TEAM │
│      1564 │         1 │   435944 │ 2023-08-12 12:00:00 │ Arsenal FC                │ Nottingham Forest FC       │ HOME_TEAM │
│      1564 │         1 │   435945 │ 2023-08-12 14:00:00 │ AFC Bournemouth           │ West Ham United FC         │ DRAW      │
│      1564 │         1 │   435946 │ 2023-08-12 14:00:00 │ Brighton & Hove Albion FC │ Luton Town FC    

In [21]:
con.sql("""
    CREATE TABLE h2h_form AS
    WITH y as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team, opponent order by date desc) as rn
        from y
    )

    select team, opponent, avg(points) as h2h_form_pl, count(*) as meetings
    from ranked
    where rn <= 6
    group by team, opponent
""")

con.sql("SELECT * FROM h2h_form").show()

┌────────────────────────────┬────────────────────────────┬────────────────────┬──────────┐
│            team            │          opponent          │    h2h_form_pl     │ meetings │
│          varchar           │          varchar           │       double       │  int64   │
├────────────────────────────┼────────────────────────────┼────────────────────┼──────────┤
│ Brighton & Hove Albion FC  │ West Ham United FC         │ 1.1666666666666667 │        6 │
│ Liverpool FC               │ Aston Villa FC             │ 1.8333333333333333 │        6 │
│ Tottenham Hotspur FC       │ Sheffield United FC        │                3.0 │        2 │
│ Newcastle United FC        │ Brentford FC               │                1.5 │        6 │
│ Everton FC                 │ Luton Town FC              │                0.5 │        2 │
│ West Ham United FC         │ Sheffield United FC        │                2.0 │        2 │
│ Wolverhampton Wanderers FC │ Manchester City FC         │                0.5 │